
## Folder 04 — File 07: Silver Inventory:


**Transformation rules:**

1. Trim inventory, product, and warehouse IDs.
2. Convert quantity_on_hand into an integer.
3. Convert snapshot_date into a date.
4. Reject missing identifiers and invalid quantities.
5. Reject negative inventory quantities.
6. Deduplicate using the combination of product_id, warehouse_id, and snapshot_date.
7. Save clean records and invalid records separately.


Important: **An inventory snapshot is identified by product, warehouse, and snapshot date. Deduplicating only by inventory_snapshot_id would not detect duplicate business snapshots with different IDs**

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_inventory_snapshots"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_inventory"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_inventory"
)


## 3. Create Silver schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


## 4. Read Bronze inventory data

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run Folder 03 first."
    )

bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze inventory records: {bronze_df.count()}")

display(bronze_df.limit(10))


## 5. Clean and standardize inventory columns

cleaned_df = (
    bronze_df

    .withColumn(
        "inventory_snapshot_id",
        F.trim(
            F.col("inventory_snapshot_id").cast("string")
        )
    )

    .withColumn(
        "product_id",
        F.trim(F.col("product_id").cast("string"))
    )

    .withColumn(
        "warehouse_id",
        F.upper(
            F.trim(F.col("warehouse_id").cast("string"))
        )
    )

    .withColumn(
        "quantity_on_hand",
        F.expr("try_cast(quantity_on_hand AS INT)")
    )

    .withColumn(
        "snapshot_date",
        F.expr("try_cast(snapshot_date AS DATE)")
    )
)

print("Inventory standardization completed.")

display(cleaned_df.limit(10))


## 6. Identify invalid inventory records

invalid_condition = (
    F.col("inventory_snapshot_id").isNull()
    | (F.col("inventory_snapshot_id") == "")

    | F.col("product_id").isNull()
    | (F.col("product_id") == "")

    | F.col("warehouse_id").isNull()
    | (F.col("warehouse_id") == "")

    | F.col("quantity_on_hand").isNull()
    | (F.col("quantity_on_hand") < 0)

    | F.col("snapshot_date").isNull()
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)
    .withColumn(
        "_quarantine_reason",
        F.concat_ws(
            "; ",

            F.when(
                F.col("inventory_snapshot_id").isNull()
                | (F.col("inventory_snapshot_id") == ""),
                F.lit("Missing inventory_snapshot_id")
            ),

            F.when(
                F.col("product_id").isNull()
                | (F.col("product_id") == ""),
                F.lit("Missing product_id")
            ),

            F.when(
                F.col("warehouse_id").isNull()
                | (F.col("warehouse_id") == ""),
                F.lit("Missing warehouse_id")
            ),

            F.when(
                F.col("quantity_on_hand").isNull(),
                F.lit("Missing or invalid quantity_on_hand")
            ),

            F.when(
                F.col("quantity_on_hand") < 0,
                F.lit("Negative inventory quantity")
            ),

            F.when(
                F.col("snapshot_date").isNull(),
                F.lit("Missing or invalid snapshot_date")
            )
        )
    )
)

valid_df = cleaned_df.filter(~invalid_condition)

print(f"Valid inventory records: {valid_df.count()}")
print(f"Invalid inventory records: {invalid_df.count()}")


## 7. Remove duplicate inventory snapshots

deduplication_window = (
    Window
    .partitionBy(
        "product_id",
        "warehouse_id",
        "snapshot_date"
    )
    .orderBy(
        F.col("_ingestion_timestamp").desc_nulls_last(),
        F.col("_source_file").desc_nulls_last(),
        F.col("inventory_snapshot_id").desc()
    )
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(deduplication_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    f"Records after deduplication: "
    f"{deduplicated_df.count()}"
)


## 8. Save invalid records to quarantine

(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table created: {QUARANTINE_TABLE}")


## 9. Save clean records to Silver Delta table

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table created: {SILVER_TABLE}")


## 10. Validate Silver inventory records

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_snapshot_groups = (
    silver_df
    .groupBy(
        "product_id",
        "warehouse_id",
        "snapshot_date"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_silver_records = (
    silver_df
    .filter(
        F.col("inventory_snapshot_id").isNull()
        | F.col("product_id").isNull()
        | F.col("warehouse_id").isNull()
        | F.col("quantity_on_hand").isNull()
        | (F.col("quantity_on_hand") < 0)
        | F.col("snapshot_date").isNull()
    )
    .count()
)

print("=" * 50)
print("SILVER INVENTORY VALIDATION")
print("=" * 50)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate snapshot groups: {duplicate_snapshot_groups}")
print(f"Invalid Silver records: {invalid_silver_records}")

if duplicate_snapshot_groups > 0 or invalid_silver_records > 0:
    raise Exception("Silver inventory validation failed.")

print("Silver inventory validation passed.")


## 11. Inventory quantity by warehouse

display(
    silver_df
    .groupBy("warehouse_id")
    .agg(
        F.count("*").alias("total_inventory_snapshots"),
        F.sum("quantity_on_hand").alias("total_quantity_on_hand"),
        F.avg("quantity_on_hand").alias("average_quantity_on_hand")
    )
    .orderBy("warehouse_id")
)


## 12. Latest inventory snapshot by product and warehouse

latest_snapshot_window = (
    Window
    .partitionBy("product_id", "warehouse_id")
    .orderBy(F.col("snapshot_date").desc())
)

latest_inventory_df = (
    silver_df
    .withColumn(
        "_row_number",
        F.row_number().over(latest_snapshot_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

display(
    latest_inventory_df
    .orderBy("product_id", "warehouse_id")
)


## 13. Identify low-stock products

LOW_STOCK_THRESHOLD = 10

low_stock_df = (
    latest_inventory_df
    .filter(
        F.col("quantity_on_hand") < LOW_STOCK_THRESHOLD
    )
)

display(
    low_stock_df.select(
        "product_id",
        "warehouse_id",
        "quantity_on_hand",
        "snapshot_date"
    )
)


## 14. Inventory quantity distribution


display(
    latest_inventory_df
    .withColumn(
        "stock_category",
        F.when(
            F.col("quantity_on_hand") == 0,
            "OUT_OF_STOCK"
        )
        .when(
            F.col("quantity_on_hand") < LOW_STOCK_THRESHOLD,
            "LOW_STOCK"
        )
        .otherwise("AVAILABLE")
    )
    .groupBy("stock_category")
    .agg(
        F.count("*").alias("product_warehouse_count"),
        F.sum("quantity_on_hand").alias("total_quantity")
    )
    .orderBy("stock_category")
)


## 15. Display final Silver inventory data

display(
    silver_df
    .orderBy("snapshot_date")
    .limit(20)
)


## 16. Final summary

print("=" * 50)
print("INVENTORY SILVER TRANSFORMATION COMPLETED")
print("=" * 50)

print(f"Bronze table     : {BRONZE_TABLE}")
print(f"Silver table     : {SILVER_TABLE}")
print(f"Quarantine table : {QUARANTINE_TABLE}")

print(f"Bronze records   : {bronze_df.count()}")
print(f"Silver records   : {total_silver_records}")
print(f"Invalid records  : {invalid_df.count()}")

print("=" * 50)



**Run this inventory verification SQL:**

In [0]:
%sql
SELECT
    'Bronze Inventory' AS layer,
    COUNT(*) AS total_records
FROM workspace.ecommerce_bronze.bronze_inventory_snapshots

UNION ALL

SELECT
    'Silver Inventory',
    COUNT(*)
FROM workspace.ecommerce_silver.silver_inventory

UNION ALL

SELECT
    'Quarantine Inventory',
    COUNT(*)
FROM workspace.ecommerce_silver.quarantine_inventory;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_inventory_status;

DESCRIBE TABLE workspace.ecommerce_gold.gold_warehouse_inventory;

DESCRIBE TABLE workspace.ecommerce_gold.gold_inventory_alerts;

DESCRIBE TABLE workspace.ecommerce_gold.gold_inventory_snapshot_trends;

DESCRIBE TABLE workspace.ecommerce_silver.data_quality_dashboard_summary;

DESCRIBE TABLE workspace.ecommerce_silver.data_quality_dashboard_detail;

DESCRIBE TABLE workspace.ecommerce_silver.data_quality_report;

DESCRIBE TABLE workspace.ecommerce_silver.quarantine_summary;

DESCRIBE TABLE workspace.ecommerce_silver.quarantine_records;

DESCRIBE TABLE workspace.ecommerce_silver.quarantine_history;

DESCRIBE TABLE workspace.ecommerce_silver.referential_integrity_dashboard_summary;

DESCRIBE TABLE workspace.ecommerce_silver.referential_integrity_dashboard_detail;

DESCRIBE TABLE workspace.ecommerce_silver.foreign_key_relationship_report;

DESCRIBE TABLE workspace.ecommerce_silver.foreign_key_cardinality_report;

DESCRIBE TABLE workspace.ecommerce_silver.orphan_record_quarantine;

DESCRIBE TABLE workspace.ecommerce_silver.orphan_records_ready_for_reprocessing;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.orphan_record_quarantine;

DESCRIBE TABLE workspace.ecommerce_silver.foreign_key_relationship_report;

DESCRIBE TABLE workspace.ecommerce_silver.foreign_key_cardinality_report;

DESCRIBE TABLE workspace.ecommerce_silver.referential_integrity_dashboard_summary;

DESCRIBE TABLE workspace.ecommerce_silver.referential_integrity_dashboard_detail;